### Final test-data preparation

Input:
`3_test_data/checkpoints/checkpoint_llm_qa_test.csv`

Final output:
`3_test_data/niletts_llm_qa_test_reference.csv`

This notebook filters existing generated QA pairs and normalizes labels.

In [ ]:
%pip install "numpy==1.26.4" "pandas==2.2.2"

In [2]:
import re
import pandas as pd


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "2_QA_GENERATION").is_dir():
        QA_DIR = candidate / "2_QA_GENERATION"
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )
# =====

In [ ]:
# =====
# CHECKPOINT_TEST_FILE = "checkpoint_llm_qa_test.csv"
# OUTPUT_TEST_FILE = "niletts_llm_qa_test_reference_filtered.csv"
# MISSING_TEST_FILE = "missing_qa_test_source_indices.csv"

CHECKPOINT_TEST_FILE = (
    QA_DIR / "3_test_data" / "checkpoints"
    / "checkpoint_llm_qa_test.csv"
)

OUTPUT_TEST_FILE = (
    QA_DIR / "3_test_data"
    / "niletts_llm_qa_test_reference.csv"
)

MISSING_TEST_FILE = (
    QA_DIR / "3_test_data" / "missing"
    / "missing_qa_test_source_indices.csv"
)
# =====

EXPECTED_TEST_ROWS = 950

In [4]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def normalize_question_type(x):
    x = normalize_spaces(x).lower()

    allowed = {
        "main_idea",
        "detail_extraction",
        "intent",
        "domain_specific",
        "number_extraction",
    }

    if x in allowed:
        return x

    parts = [p.strip() for p in x.split("/") if p.strip()]

    for preferred in [
        "domain_specific",
        "detail_extraction",
        "intent",
        "main_idea",
        "number_extraction",
    ]:
        if preferred in parts:
            return preferred

    if "price" in x or "number" in x:
        return "number_extraction"

    return "other"


def normalize_difficulty(x):
    x = normalize_spaces(x).lower()

    if x in {"easy", "medium"}:
        return x

    if x == "hard":
        return "medium"

    return "medium"


In [5]:
test_df = pd.read_csv(CHECKPOINT_TEST_FILE)

print("Raw test checkpoint shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())


Raw test checkpoint shape: (2818, 14)
Columns: ['id', 'split', 'source_row_index', 'audio_file', 'speaker_name', 'domain', 'context_ref', 'question', 'answer', 'answer_start', 'question_type', 'difficulty', 'valid_answer_span', 'needs_manual_review']


In [ ]:
clean_df = test_df.copy()

for col in ["question", "answer", "context_ref", "audio_file", "speaker_name", "domain"]:
    clean_df[col] = clean_df[col].astype(str).map(normalize_spaces)

# Recompute answer position inside reference transcript
clean_df["answer_start_ref"] = clean_df.apply(
    lambda r: find_answer_start_soft(r["context_ref"], r["answer"]),
    axis=1
)

clean_df["valid_answer_span_ref"] = clean_df["answer_start_ref"] != -1

# Keep original raw labels before cleaning
clean_df["question_type_raw"] = clean_df["question_type"].astype(str)
clean_df["difficulty_raw"] = clean_df["difficulty"].astype(str)

# Normalize labels
clean_df["question_type"] = clean_df["question_type"].map(normalize_question_type)
clean_df["difficulty"] = clean_df["difficulty"].map(normalize_difficulty)

before = len(clean_df)

clean_df = clean_df[clean_df["question"].str.len() > 5]
clean_df = clean_df[clean_df["answer"].str.len() > 0]
clean_df = clean_df[clean_df["valid_answer_span_ref"] == True]

clean_df = clean_df.drop_duplicates(
    subset=["audio_file", "question", "answer"]
).reset_index(drop=True)

print("Rows before filtering:", before)
print("Rows after filtering:", len(clean_df))
print("Removed rows:", before - len(clean_df))


Rows before filtering: 2818
Rows after filtering: 1554
Removed rows: 1264


In [8]:
covered_source_rows = clean_df["source_row_index"].nunique()
covered_indices = set(clean_df["source_row_index"].astype(int).tolist())
missing_indices = sorted(list(set(range(EXPECTED_TEST_ROWS)) - covered_indices))

print("Expected test source rows:", EXPECTED_TEST_ROWS)
print("Covered test source rows after filtering:", covered_source_rows)
print("Missing test source rows after filtering:", len(missing_indices))
print("First 30 missing indices:", missing_indices[:30])


Expected test source rows: 950
Covered test source rows after filtering: 738
Missing test source rows after filtering: 212
First 30 missing indices: [7, 20, 26, 29, 34, 39, 47, 48, 52, 57, 64, 76, 77, 81, 95, 102, 106, 115, 116, 119, 120, 121, 129, 131, 134, 139, 140, 145, 153, 155]


In [9]:
print("\nQuestion type distribution:")
display(clean_df["question_type"].value_counts(dropna=False))



Question type distribution:


question_type
detail_extraction    551
domain_specific      492
intent               322
main_idea            162
number_extraction     27
Name: count, dtype: int64

In [10]:
print("\nDifficulty distribution:")
display(clean_df["difficulty"].value_counts(dropna=False))



Difficulty distribution:


difficulty
easy      1044
medium     510
Name: count, dtype: int64

In [11]:
print("\nDomain distribution:")
display(clean_df["domain"].value_counts(dropna=False))



Domain distribution:


domain
sales      847
general    495
medical    212
Name: count, dtype: int64

In [ ]:
# Rename source_row_index to qa_test_source_index for clarity
clean_df = clean_df.rename(columns={
    "source_row_index": "qa_test_source_index"
})

# Keep useful columns in clean order
final_cols = [
    "id",
    "split",
    "qa_test_source_index",
    "audio_file",
    "speaker_name",
    "domain",
    "context_ref",
    "question",
    "answer",
    "answer_start_ref",
    "valid_answer_span_ref",
    "question_type",
    "question_type_raw",
    "difficulty",
    "difficulty_raw",
    "needs_manual_review"
]

final_cols = [c for c in final_cols if c in clean_df.columns]
final_test_df = clean_df[final_cols].copy()


In [13]:
final_test_df.to_csv(
    OUTPUT_TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "qa_test_source_index": missing_indices
}).to_csv(
    MISSING_TEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", OUTPUT_TEST_FILE)
print("Saved:", MISSING_TEST_FILE)
print("Final test shape:", final_test_df.shape)

display(final_test_df.head())


Saved: niletts_llm_qa_test_reference_filtered.csv
Saved: missing_qa_test_source_indices.csv
Final test shape: (1554, 16)


,id,split,qa_test_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,question_type_raw,difficulty,difficulty_raw,needs_manual_review
0,test_0_0,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,الشخص الذي يُعتبر حافزًا في هذا السياق هو من ي...,الشخص اللي هو يعني حافز مش فاهم ده.,1,True,domain_specific,domain_specific,easy,easy,True
1,test_0_1,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,ما هي المشكلة التي قد تنشأ بسبب الفضول؟,الأهم دلوقتي الشخص اللي عنده فضول.,105,True,intent,intent/main_idea,medium,medium,True
2,test_0_2,test,0,wavs/sales_34_00019.wav,SPEAKER_02,sales,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,هل هناك أي مشكلة معقدة يمكن أن تنشأ بسبب الفضول؟,ده ممكن يبقى مشكلة بعدين.,79,True,detail_extraction,detail_extraction,easy,easy,True
3,test_1_0,test,1,wavs/sales_26_00082.wav,SPEAKER_01,sales,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,ما هو الموضوع الأهم الذي سيتم التطرق إليه خلال...,بناء فريق عمل ناجح.,57,True,main_idea,main_idea,easy,easy,True
4,test_1_1,test,1,wavs/sales_26_00082.wav,SPEAKER_01,sales,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,هل هناك أي موضوع آخر سيُستكشف بعد ذلك؟,موضوع تاني لا يقل أهمية أبداً.,117,True,detail_extraction,detail_extraction,medium,medium,True
